# Optuna Results Viewer
PyCharmのこのセルを実行すると、Optunaのデータベースから結果を取得し、インタラクティブなテーブルとして表示します。
`target_study_id` または `target_study_name` を変更することで、動的に読み込むStudyを切り替えられます。

In [18]:
import optuna
import pandas as pd
import sqlite3

# ==========================================
# 設定: ここを書き換えて動的にStudyを指定します
# ==========================================
db_path = "optuna_qrc_nonetype_mean_vpt.db"

# Studyの指定方法（どちらかを使用。両方Noneなら一番最新を取得します）
target_study_id = 5    # 例: 1 (データベース内のIDで指定する場合)
target_study_name = None  # 例: "qrc_lorenz_vpt..." (名前で指定する場合)

# フィルタリング設定: value（VPTなど）がこの値以上の行のみ表示します
min_value_threshold = 1.0 # 例: 5.0
# ==========================================

storage_url = f"sqlite:///{db_path}"

# 利用可能なStudy一覧を取得するヘルパー関数
def get_studies(db_file):
    try:
        with sqlite3.connect(db_file) as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT study_id, study_name FROM studies ORDER BY study_id ASC")
            return cursor.fetchall()
    except Exception as e:
        return []

studies = get_studies(db_path)

if not studies:
    print(f"データベース '{db_path}' にStudyが見つからない、またはファイルが存在しません。")
    df_view = pd.DataFrame({"Error": ["Studyが見つかりません"]})
else:
    print("【利用可能なStudy一覧】")
    for s_id, s_name in studies:
        print(f"  ID: {s_id} | Name: {s_name}")
        
    # どのStudyを読み込むか決定
    selected_name = None
    if target_study_id is not None:
        for s_id, s_name in studies:
            if s_id == target_study_id:
                selected_name = s_name
                break
        if selected_name is None:
            print(f"\n⚠️ 警告: 指定された ID '{target_study_id}' は見つかりませんでした。")
            
    elif target_study_name is not None:
        for s_id, s_name in studies:
            if s_name == target_study_name:
                selected_name = s_name
                break
        if selected_name is None:
            print(f"\n⚠️ 警告: 指定された Name '{target_study_name}' は見つかりませんでした。")

    # 指定がない、または見つからなかった場合は最新（最後）のものを選択
    if selected_name is None:
        selected_name = studies[-1][1]
        print(f"\n-> 🔄 最新のStudyを読み込みます: {selected_name}")
    else:
        print(f"\n-> ✅ 指定されたStudyを読み込みます: {selected_name}")

    # Optunaからデータを取得
    study = optuna.load_study(study_name=selected_name, storage=storage_url)
    df = study.trials_dataframe()
    
    # 見やすいようにフォーマット（COMPLETEのみ抽出し、VPTで降順ソート）
    if 'state' in df.columns:
        df = df[df['state'] == 'COMPLETE']
    if 'value' in df.columns:
        # 指定された閾値以上のものだけをフィルタリング
        if min_value_threshold is not None:
            df = df[df['value'] >= min_value_threshold]
        df = df.sort_values(by='value', ascending=False)
        
    # 不要な列を隠して見やすくする（datetime等）
    drop_cols = [c for c in df.columns if c.startswith('datetime_') or c.startswith('duration')]
    df_view = df.drop(columns=drop_cols)
    
    # 必要な列を抽出してCSVに保存
    import os
    # 'value' と 'params_' で始まるすべての列を抽出
    export_cols = ['value'] + [c for c in df_view.columns if c.startswith('params_')]
    valid_export_cols = [c for c in export_cols if c in df_view.columns]
    
    if valid_export_cols:
        out_path = os.path.join("/home/yoshi/PycharmProjects/Reservoir/benchmarks", "filtered_optuna_results.csv")
        df_view[valid_export_cols].to_csv(out_path, index=False)
        print(f"\n✅ {len(valid_export_cols)-1} 個のパラメータを含むデータを '{out_path}' に保存しました。")

# DataFrameを表示（PyCharmが綺麗なテーブルでレンダリングします）
df_view.head(50)

【利用可能なStudy一覧】
  ID: 1 | Name: qrc_lorenz_vpt_minmax0_nonetype_q5_Z+ZZ_poly_square_reupTrue_mean_vpt_fb<2
  ID: 2 | Name: qrc_lorenz_vpt_minmax0_nonetype_q10_Z+ZZ_poly_square_reupTrue_mean_vpt_fb<2
  ID: 3 | Name: qrc_lorenz_vpt_minmax0_nonetype_q11_Z+ZZ_poly_square_reupTrue_mean_vpt_fb<2
  ID: 4 | Name: qrc_mackey_glass_vpt_bounded_affine0_nonetype_q6_Z+ZZ_poly_square_reupTrue_mean_vpt_fb<3.5
  ID: 5 | Name: qrc_mackey_glass_vpt_bounded_affine0_nonetype_q10_Z+ZZ_poly_square_reupTrue_mean_vpt_fb<3.5
  ID: 6 | Name: qrc_mackey_glass_vpt_bounded_affine0_nonetype_q8_Z+ZZ_poly_square_reupTrue_mean_vpt_fb<3.5
  ID: 7 | Name: qrc_mackey_glass_vpt_bounded_affine0_nonetype_q11_Z+ZZ_poly_square_reupTrue_mean_vpt_fb<3.5

-> ✅ 指定されたStudyを読み込みます: qrc_mackey_glass_vpt_bounded_affine0_nonetype_q10_Z+ZZ_poly_square_reupTrue_mean_vpt_fb<3.5

✅ 5 個のパラメータを含むデータを '/home/yoshi/PycharmProjects/Reservoir/benchmarks/filtered_optuna_results.csv' に保存しました。


,number,value,params_feedback_scale,params_leak_rate,params_n_layers,params_relative_shift,params_scale,user_attrs_best_lambda_seed40,user_attrs_best_lambda_seed41,user_attrs_best_lambda_seed42,...,user_attrs_vpt_lt_seed42,user_attrs_vpt_lt_seed43,user_attrs_vpt_lt_seed44,user_attrs_vpt_steps_seed40,user_attrs_vpt_steps_seed41,user_attrs_vpt_steps_seed42,user_attrs_vpt_steps_seed43,user_attrs_vpt_steps_seed44,system_attrs_fixed_params,state
30,30,1.765060,2.353620,0.170919,1,0.039626,0.038116,4.520354e-08,4.520354e-08,4.520354e-08,...,1.289157,0.644578,1.283133,770.0,161.0,214.0,107.0,213.0,"{'feedback_scale': 2.3536203916302614, 'leak_r...",COMPLETE
46,46,1.645783,2.572491,0.160941,1,0.045579,0.043592,4.520354e-08,4.520354e-08,1.487352e-07,...,1.174699,0.855422,1.168675,522.0,313.0,195.0,142.0,194.0,"{'feedback_scale': 2.5724914895966395, 'leak_r...",COMPLETE
37,37,1.600000,2.382770,0.161709,1,0.042169,0.040463,4.520354e-08,4.520354e-08,1.487352e-07,...,1.289157,0.584337,1.289157,547.0,256.0,214.0,97.0,214.0,"{'feedback_scale': 2.3827704241631786, 'leak_r...",COMPLETE
26,26,1.583133,2.565261,0.156442,1,0.044850,0.042924,4.520354e-08,4.520354e-08,1.487352e-07,...,1.397590,0.855422,1.162651,435.0,312.0,232.0,142.0,193.0,"{'feedback_scale': 2.565261048592284, 'leak_ra...",COMPLETE
22,22,1.432530,2.506132,0.192225,1,0.038606,0.037171,4.520354e-08,4.520354e-08,4.520354e-08,...,1.403614,1.638554,1.289157,308.0,162.0,233.0,272.0,214.0,"{'feedback_scale': 2.506131764698665, 'leak_ra...",COMPLETE
19,19,1.213253,2.519889,0.160841,1,0.040896,0.039289,4.520354e-08,4.520354e-08,4.520354e-08,...,1.415663,0.632530,1.174699,215.0,257.0,235.0,105.0,195.0,"{'feedback_scale': 2.519889140039358, 'leak_ra...",COMPLETE
12,12,1.208434,2.405064,0.173660,1,0.036354,0.035079,4.520354e-08,4.520354e-08,4.520354e-08,...,1.746988,1.295181,1.162651,144.0,161.0,290.0,215.0,193.0,"{'feedback_scale': 2.4050644466185203, 'leak_r...",COMPLETE
14,14,1.184337,2.551285,0.165352,1,0.033758,0.032656,4.520354e-08,4.520354e-08,4.520354e-08,...,1.307229,1.746988,1.174699,119.0,162.0,217.0,290.0,195.0,"{'feedback_scale': 2.5512851485670938, 'leak_r...",COMPLETE
24,24,1.165060,2.319718,0.166033,1,0.038852,0.037399,4.520354e-08,4.520354e-08,4.520354e-08,...,1.403614,0.620482,1.283133,257.0,161.0,233.0,103.0,213.0,"{'feedback_scale': 2.3197183061957025, 'leak_r...",COMPLETE
17,17,1.118072,2.261380,0.146658,1,0.039681,0.038167,4.520354e-08,1.487352e-07,1.487352e-07,...,1.518072,0.301205,0.909639,313.0,162.0,252.0,50.0,151.0,"{'feedback_scale': 2.261380136423591, 'leak_ra...",COMPLETE
